# EconomicsLab -- Comprehensive Tutorial

**An interactive journey through applied econometrics**

---

## Table of Contents

1. [Introduction & Setup](#1.-Introduction-&-Setup)
2. [Descriptive Statistics & Visualization](#2.-Descriptive-Statistics-&-Visualization)
3. [Ordinary Least Squares (OLS)](#3.-Ordinary-Least-Squares-(OLS))
4. [Instrumental Variables (IV / 2SLS)](#4.-Instrumental-Variables-(IV-/-2SLS))
5. [Difference-in-Differences (DiD)](#5.-Difference-in-Differences-(DiD))
6. [Panel Data & Fixed Effects](#6.-Panel-Data-&-Fixed-Effects)
7. [Binary Choice Models (Logit / Probit)](#7.-Binary-Choice-Models-(Logit-/-Probit))
8. [Regression Discontinuity Design (RDD)](#8.-Regression-Discontinuity-Design-(RDD))
9. [LASSO for Variable Selection](#9.-LASSO-for-Variable-Selection)
10. [Bootstrap Inference](#10.-Bootstrap-Inference)

---

**Prerequisites:** Basic knowledge of Python, statistics, and econometrics.

**References:** Each section cites the original papers that introduced or popularized the method.

## 1. Introduction & Setup

In [ ]:
# Core imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context('notebook', font_scale=1.1)
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100

# EconomicsLab
import econlab
from econlab.datasets import (
    load_nlsy,
    load_angrist_krueger,
    load_card_minwage,
    list_datasets,
    get_dataset_info,
)

# Stats libraries
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.iolib.summary2 import summary_col

print(f"EconomicsLab version: {econlab.__version__}")
print(f"Available datasets: {list_datasets()}")

### Dataset Overview

EconomicsLab ships with three classic economics datasets. Let's explore each one.

In [ ]:
for name in list_datasets():
    info = get_dataset_info(name)
    print(f"\n{'='*60}")
    print(f"Dataset: {info['name']}")
    print(f"  Description: {info['description'][:80]}...")
    print(f"  Observations: {info['n_obs']:,}")
    print(f"  Variables: {info['n_vars']}")
    print(f"  Reference: {info['reference']}")

## 2. Descriptive Statistics & Visualization

In [ ]:
# Load NLSY panel data
nlsy = load_nlsy()
print(f"Shape: {nlsy.shape}")
print(f"Years: {sorted(nlsy['year'].unique())}")
print(f"Individuals: {nlsy['id'].nunique()}")
nlsy.head(10)

In [ ]:
# Summary statistics
nlsy.describe().round(3)

In [ ]:
# Wage distribution by gender
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(data=nlsy, x='log_wage', hue='female', bins=40, kde=True, ax=axes[0])
axes[0].set_title('Log Wage Distribution by Gender')
axes[0].set_xlabel('log(Wage)')

sns.boxplot(data=nlsy, x='female', y='log_wage', ax=axes[1])
axes[1].set_title('Log Wage by Gender')
axes[1].set_xticklabels(['Male', 'Female'])

plt.tight_layout()
plt.show()

In [ ]:
# Education-wage relationship (Mincer)
fig, ax = plt.subplots(figsize=(10, 6))
sns.scatterplot(data=nlsy.sample(500), x='education', y='log_wage', alpha=0.3, ax=ax)
sns.regplot(data=nlsy.sample(500), x='education', y='log_wage', 
            scatter=False, color='red', line_kws={'linewidth': 2}, ax=ax)
ax.set_title('Returns to Education (Sample of 500 obs)')
ax.set_xlabel('Years of Education')
ax.set_ylabel('log(Wage)')
plt.show()

In [ ]:
# Correlation heatmap
numeric_cols = ['age', 'education', 'experience', 'tenure', 'log_wage', 'wage', 'hours']
corr = nlsy[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, linewidths=0.5, ax=ax)
ax.set_title('Correlation Matrix of NLSY Variables')
plt.show()

## 3. Ordinary Least Squares (OLS)

**The Mincer earnings equation** (Mincer, 1974) is the most famous application of OLS in labor economics:

$$\ln(w_i) = \beta_0 + \beta_1 \cdot \text{educ}_i + \beta_2 \cdot \text{exper}_i + \beta_3 \cdot \text{exper}_i^2 + \epsilon_i$$

**Reference:** Mincer, J. (1974). *Schooling, Experience, and Earnings*. NBER.

In [ ]:
# Mincer regression using formula API
model_ols = smf.ols(
    'log_wage ~ education + experience + I(experience**2) + female',
    data=nlsy
)
results_ols = model_ols.fit(cov_type='HC1')  # Heteroskedasticity-robust SE
print(results_ols.summary())

In [ ]:
# Compare: Classical SE vs Robust SE vs Clustered SE
models_ols = {
    'Classical': model_ols.fit(),
    'HC1 Robust': model_ols.fit(cov_type='HC1'),
    'Clustered (id)': model_ols.fit(
        cov_type='cluster', cov_kwds={'groups': nlsy['id']}
    ),
}

print(summary_col(
    list(models_ols.values()),
    stars=True,
    model_names=list(models_ols.keys()),
    info_dict={'N': lambda x: f"{int(x.nobs):,}", 'R2': lambda x: f"{x.rsquared:.4f}"},
))

### Key Takeaway

- Each additional year of education is associated with ~8% higher wages
- Experience has diminishing returns (positive linear, negative quadratic)
- Gender wage gap persists even after controlling for education and experience

## 4. Instrumental Variables (IV / 2SLS)

**The Angrist-Krueger QOB instrument** (Angrist & Krueger, 1991) uses quarter of birth as an instrument for education. The intuition: compulsory schooling laws create a relationship between birth quarter and total years of education.

$$\text{Stage 1: } \text{educ}_i = \pi_0 + \pi_1 \text{QOB}_i + \nu_i$$
$$\text{Stage 2: } \ln(w_i) = \beta_0 + \beta_1 \widehat{\text{educ}}_i + \epsilon_i$$

**Reference:** Angrist, J. D. & Krueger, A. B. (1991). *QJE*, 106(4), 979-1014.

In [ ]:
ak = load_angrist_krueger()
print(f"Shape: {ak.shape}")
ak.head()

In [ ]:
# Visualize the first stage: QOB vs Education
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=ak, x='qob', y='education', ax=axes[0])
axes[0].set_title('First Stage: Education by Quarter of Birth')
axes[0].set_xlabel('Quarter of Birth')

qob_means = ak.groupby('qob')['education'].mean()
axes[1].bar(qob_means.index, qob_means.values, color='steelblue', edgecolor='white')
axes[1].set_title('Mean Education by QOB')
axes[1].set_xlabel('Quarter of Birth')
axes[1].set_ylabel('Mean Years of Education')

plt.tight_layout()
plt.show()

In [ ]:
# First stage regression
first_stage = smf.ols('education ~ C(qob)', data=ak).fit()
print("First Stage:")
print(f"  F-statistic for QOB: {first_stage.fvalue:.2f}")
print(f"  p-value: {first_stage.f_pvalue:.4f}")
print(f"  R-squared: {first_stage.rsquared:.4f}")
print()

# Naive OLS (biased)
ols_ak = smf.ols('log_wage ~ education', data=ak).fit()
print("Naive OLS:")
print(f"  Returns to education: {ols_ak.params['education']:.4f}")
print()

# IV/2SLS using linearmodels
from linearmodels.iv import IV2SLS
import pandas as pd

qob_dummies = pd.get_dummies(ak['qob'], prefix='qob', drop_first=True)

iv_model = IV2SLS(
    dependent=ak['log_wage'],
    exog=None,
    endog=ak['education'],
    instruments=qob_dummies,
)
iv_results = iv_model.fit()
print("IV/2SLS:")
print(f"  Returns to education: {iv_results.params['education']:.4f}")
print(f"  First-stage F: {iv_results.first_stage.diagnostics['f.stat']:.2f}")

### Key Takeaway

- OLS may be biased if education is endogenous (ability bias, measurement error)
- IV using QOB yields a higher returns-to-education estimate, consistent with Angrist & Krueger (1991)
- First-stage F-statistic > 10 suggests a strong instrument (rule of thumb by Staiger & Stock, 1997)

## 5. Difference-in-Differences (DiD)

**Card & Krueger (1994)** studied the effect of New Jersey's 1992 minimum wage increase on fast-food employment, using eastern Pennsylvania as a control group.

$$\text{FTE}_{it} = \alpha + \beta_1 \text{NJ}_i + \beta_2 \text{Post}_t + \tau (\text{NJ}_i \times \text{Post}_t) + \epsilon_{it}$$

The coefficient $\tau$ is the DiD estimate of the minimum wage effect.

**Reference:** Card, D. & Krueger, A. B. (1994). *AER*, 84(4), 772-793.

In [ ]:
card = load_card_minwage()
print(f"Shape: {card.shape}")
card.head()

In [ ]:
# DiD means table
did_table = card.groupby(['state', 'time_period'])['fte'].agg(['mean', 'std', 'count']).round(2)
print("DiD Means Table:")
print(did_table)
print()

# Manual DiD calculation
means = card.groupby(['state', 'time_period'])['fte'].mean()
nj_before = means[('NJ', 'before')]
nj_after = means[('NJ', 'after')]
pa_before = means[('PA', 'before')]
pa_after = means[('PA', 'after')]

did_manual = (nj_after - nj_before) - (pa_after - pa_before)
print(f"\nManual DiD Estimate: {did_manual:.3f}")

In [ ]:
# DiD regression
did_model = smf.ols('fte ~ treated + post + treated:post', data=card)
did_results = did_model.fit(cov_type='HC1')
print(did_results.summary())

In [ ]:
# Visualization: Parallel trends
fig, ax = plt.subplots(figsize=(10, 6))

for state, color, marker in [('NJ', 'blue', 'o'), ('PA', 'red', 's')]:
    subset = card[card['state'] == state]
    means_plot = subset.groupby('time_period')['fte'].mean()
    ax.plot([0, 1], means_plot.values, color=color, marker=marker,
            markersize=12, linewidth=2.5, label=state)

ax.set_xticks([0, 1])
ax.set_xticklabels(['Before', 'After'])
ax.set_ylabel('Mean FTE Employment')
ax.set_title('Card & Krueger (1994): Minimum Wage and Employment')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

### Key Takeaway

- The DiD estimate ($\tau$) captures the causal effect of the minimum wage increase
- The parallel trends assumption requires that NJ and PA would have followed similar employment trends absent the policy change
- Card & Krueger found no negative employment effect, challenging the standard competitive model

## 6. Panel Data & Fixed Effects

**Panel fixed effects** remove time-invariant unobserved heterogeneity by demeaning each variable within entity. This controls for all individual-specific confounders that do not vary over time.

$$y_{it} - \bar{y}_i = \beta (X_{it} - \bar{X}_i) + (\epsilon_{it} - \bar{\epsilon}_i)$$

**Reference:** Wooldridge, J. M. (2010). *Econometric Analysis of Cross Section and Panel Data*. MIT Press.

In [ ]:
# Prepare panel data
panel_df = nlsy.copy()
panel_df = panel_df.set_index(['id', 'year'])
print(f"Panel dimensions: {panel_df.index.levshape}")
print(f"Balanced panel: {panel_df.index.levshape[0] * panel_df.index.levshape[1] == len(panel_df)}")

In [ ]:
from linearmodels.panel import PanelOLS, RandomEffects

# Pooled OLS (ignoring panel structure)
pooled = PanelOLS(
    dependent=panel_df['log_wage'],
    exog=panel_df[['experience', 'hours', 'union']],
    entity_effects=False,
)
pooled_results = pooled.fit()

# Fixed Effects (entity demeaning)
fe = PanelOLS(
    dependent=panel_df['log_wage'],
    exog=panel_df[['experience', 'hours', 'union']],
    entity_effects=True,
)
fe_results = fe.fit()

print("="*60)
print("Pooled OLS")
print("="*60)
print(pooled_results.summary)
print()
print("="*60)
print("Fixed Effects")
print("="*60)
print(fe_results.summary)

In [ ]:
# Fixed Effects with time trends
panel_df_fe = panel_df.copy()
panel_df_fe['year_trend'] = panel_df_fe.index.get_level_values('year') - 1979

fe_trend = PanelOLS(
    dependent=panel_df_fe['log_wage'],
    exog=panel_df_fe[['experience', 'hours', 'union', 'year_trend']],
    entity_effects=True,
)
fe_trend_results = fe_trend.fit()

print("Fixed Effects with Time Trend:")
print(fe_trend_results.summary)

### Key Takeaway

- Fixed effects estimates can differ substantially from pooled OLS when there is unobserved individual heterogeneity
- The Hausman test can help choose between fixed effects and random effects
- Standard errors should be clustered at the entity level in panel data

## 7. Binary Choice Models (Logit / Probit)

**Binary choice models** estimate the probability of a binary outcome as a function of covariates.

$$P(Y_i = 1 | X_i) = F(X_i\beta)$$

where $F$ is the logistic CDF (Logit) or normal CDF (Probit).

**Reference:** McFadden, D. (1974). "Conditional Logit Analysis of Qualitative Choice Behavior." In *Frontiers in Econometrics*.

In [ ]:
# Prepare data: model union membership
X_logit = sm.add_constant(nlsy[['education', 'experience', 'age', 'female']])
y_logit = nlsy['union']

# Logit
logit_model = sm.Logit(y_logit, X_logit)
logit_results = logit_model.fit(disp=False)

# Probit
probit_model = sm.Probit(y_logit, X_logit)
probit_results = probit_model.fit(disp=False)

print("="*60)
print("Logit Results")
print("="*60)
print(logit_results.summary())
print()
print("="*60)
print("Probit Results")
print("="*60)
print(probit_results.summary())

In [ ]:
# Marginal effects at the mean (AME)
logit_mfx = logit_results.get_margeff(at='mean')
print("Logit Marginal Effects (at mean):")
print(logit_mfx.summary())

In [ ]:
# Compare Logit vs Probit fitted probabilities
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(logit_results.predict(), probit_results.predict(), alpha=0.3, s=10)
ax.plot([0, 1], [0, 1], 'r--', linewidth=1, label='45-degree line')
ax.set_xlabel('Logit Predicted Probability')
ax.set_ylabel('Probit Predicted Probability')
ax.set_title('Logit vs Probit: Predicted Probabilities')
ax.legend()
plt.show()

### Key Takeaway

- Logit and Probit typically produce very similar predicted probabilities and marginal effects
- Coefficients are NOT directly comparable across models -- use marginal effects for comparison
- Logit has fatter tails; Probit assumes normality

## 8. Regression Discontinuity Design (RDD)

**RDD** exploits a discontinuous change in treatment assignment at a known cutoff to estimate causal effects.

$$Y_i = \alpha + \tau D_i + \beta_1 (X_i - c) + \beta_2 D_i \cdot (X_i - c) + \epsilon_i$$

where $D_i = 1[X_i \geq c]$ indicates treatment assignment at cutoff $c$.

**Reference:** Thistlethwaite, D. L. & Campbell, D. T. (1960). *Journal of Educational Psychology*, 51(6), 309-317.

In [ ]:
# Simulate an RDD dataset
np.random.seed(42)
n_rd = 500

# Running variable: centered at 0, cutoff = 0
running_var = np.random.uniform(-50, 50, n_rd)
treatment = (running_var >= 0).astype(int)

# Outcome with treatment effect of 3.0 at cutoff
outcome = (
    10.0
    + 0.2 * running_var               # pre-treatment trend
    + 3.0 * treatment                  # treatment effect (tau)
    + 0.1 * running_var * treatment    # different slope post-cutoff
    + np.random.normal(0, 2, n_rd)     # noise
)

rd_df = pd.DataFrame({
    'running': running_var,
    'treated': treatment,
    'outcome': outcome,
})

In [ ]:
# RDD Regression (local linear)
rd_model = smf.ols('outcome ~ running + treated + running:treated', data=rd_df)
rd_results = rd_model.fit(cov_type='HC1')
print(rd_results.summary())
print(f"\nRDD Treatment Effect (tau): {rd_results.params['treated']:.3f}")

In [ ]:
# Visualization
fig, ax = plt.subplots(figsize=(12, 6))

# Scatter
colors = ['#2ecc71' if t == 0 else '#e74c3c' for t in rd_df['treated']]
ax.scatter(rd_df['running'], rd_df['outcome'], c=colors, alpha=0.4, s=20)

# Fitted lines for each side
for t_val, color, label in [(0, '#2ecc71', 'Control'), (1, '#e74c3c', 'Treated')]:
    subset = rd_df[rd_df['treated'] == t_val]
    x_range = np.linspace(subset['running'].min(), subset['running'].max(), 100)
    X_pred = pd.DataFrame({'running': x_range, 'treated': t_val})
    X_pred['running:treated'] = X_pred['running'] * X_pred['treated']
    y_pred = rd_results.predict(sm.add_constant(X_pred[['running', 'treated', 'running:treated']]))
    ax.plot(x_range, y_pred, color=color, linewidth=2.5, label=label)

ax.axvline(x=0, color='black', linestyle='--', linewidth=1.5, label='Cutoff')
ax.set_xlabel('Running Variable (centered at cutoff)')
ax.set_ylabel('Outcome')
ax.set_title('Regression Discontinuity Design: Treatment Effect at Cutoff')
ax.legend()
plt.show()

### Key Takeaway

- RDD estimates are valid under the assumption that units cannot precisely manipulate the running variable
- The bandwidth choice involves a bias-variance tradeoff
- Modern approaches use local polynomial regression with data-driven bandwidth selection (Calonico, Cattaneo, Titiunik)

## 9. LASSO for Variable Selection

**LASSO** (Least Absolute Shrinkage and Selection Operator) performs both regularization and variable selection by adding an $L_1$ penalty to the OLS objective:

$$\min_\beta \left\{ \frac{1}{2n} \|y - X\beta\|_2^2 + \lambda \|\beta\|_1 \right\}$$

**Reference:** Tibshirani, R. (1996). "Regression Shrinkage and Selection via the Lasso." *JRSS-B*, 58(1), 267-288.

In [ ]:
from sklearn.linear_model import Lasso, LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Prepare many potential predictors
nlsy_sample = nlsy.sample(1000, random_state=42)

X_lasso = nlsy_sample[['education', 'experience', 'age', 'female', 'hours', 
                        'tenure', 'union']].copy()
X_lasso['exp_sq'] = X_lasso['experience'] ** 2
X_lasso['age_sq'] = X_lasso['age'] ** 2
X_lasso['edu_exp'] = X_lasso['education'] * X_lasso['experience']

y_lasso = nlsy_sample['log_wage']

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_lasso)
X_scaled = pd.DataFrame(X_scaled, columns=X_lasso.columns)

print(f"Number of predictors: {X_scaled.shape[1]}")

In [ ]:
# Cross-validated LASSO
lasso_cv = LassoCV(cv=10, random_state=42, max_iter=10000, alphas=np.logspace(-4, 1, 50))
lasso_cv.fit(X_scaled, y_lasso)

print(f"Optimal alpha: {lasso_cv.alpha_:.4f}")
print(f"\nLASSO Coefficients:")
for name, coef in zip(X_scaled.columns, lasso_cv.coef_):
    if abs(coef) > 1e-4:
        print(f"  {name:15s}: {coef:+.4f}")
    else:
        print(f"  {name:15s}: 0 (dropped)")

In [ ]:
# LASSO path visualization
alphas = np.logspace(-3, 1, 100)
coefs = []

for alpha in alphas:
    lasso = Lasso(alpha=alpha, max_iter=10000, random_state=42)
    lasso.fit(X_scaled, y_lasso)
    coefs.append(lasso.coef_)

coefs = np.array(coefs)

fig, ax = plt.subplots(figsize=(12, 7))
for i, col in enumerate(X_scaled.columns):
    ax.plot(alphas, coefs[:, i], label=col, linewidth=1.5)

ax.set_xscale('log')
ax.set_xlabel('Alpha (regularization strength)')
ax.set_ylabel('Coefficient Value')
ax.set_title('LASSO Coefficient Paths')
ax.axvline(x=lasso_cv.alpha_, color='red', linestyle='--', label=f'Optimal alpha = {lasso_cv.alpha_:.4f}')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Key Takeaway

- LASSO automatically selects relevant variables by shrinking irrelevant coefficients to exactly zero
- Cross-validation is essential for choosing the regularization parameter $\lambda$
- In high-dimensional settings, LASSO can outperform OLS in prediction
- Post-LASSO OLS (using only selected variables) can reduce the shrinkage bias

## 10. Bootstrap Inference

**The bootstrap** (Efron, 1979) provides a powerful non-parametric method for computing standard errors, confidence intervals, and hypothesis tests without relying on asymptotic approximations.

**Reference:** Efron, B. (1979). "Bootstrap Methods: Another Look at the Jackknife." *Annals of Statistics*, 7(1), 1-26.

In [ ]:
def bootstrap_ols(data, formula, n_boot=1000, seed=42):
    """Bootstrap standard errors for OLS coefficients."""
    np.random.seed(seed)
    n = len(data)
    
    # Full-sample estimate
    full_model = smf.ols(formula, data=data).fit()
    
    # Bootstrap
    boot_coefs = []
    for _ in range(n_boot):
        boot_sample = data.sample(n=n, replace=True)
        boot_model = smf.ols(formula, data=boot_sample).fit()
        boot_coefs.append(boot_model.params.values)
    
    boot_coefs = np.array(boot_coefs)
    boot_se = boot_coefs.std(axis=0)
    
    # Percentile confidence intervals
    ci_lower = np.percentile(boot_coefs, 2.5, axis=0)
    ci_upper = np.percentile(boot_coefs, 97.5, axis=0)
    
    return {
        'params': full_model.params,
        'boot_se': pd.Series(boot_se, index=full_model.params.index),
        'ci_lower': pd.Series(ci_lower, index=full_model.params.index),
        'ci_upper': pd.Series(ci_upper, index=full_model.params.index),
        'boot_dist': boot_coefs,
    }

# Run bootstrap on Mincer equation
boot_results = bootstrap_ols(nlsy.sample(500), 
                              'log_wage ~ education + experience + female',
                              n_boot=1000)

print("Bootstrap Results for Returns to Education:")
print(f"  Coefficient: {boot_results['params']['education']:.4f}")
print(f"  Bootstrap SE: {boot_results['boot_se']['education']:.4f}")
print(f"  95% CI: [{boot_results['ci_lower']['education']:.4f}, "
      f"{boot_results['ci_upper']['education']:.4f}]")

In [ ]:
# Visualize bootstrap distribution for education coefficient
fig, ax = plt.subplots(figsize=(10, 6))

edu_idx = list(boot_results['params'].index).index('education')
edu_boot = boot_results['boot_dist'][:, edu_idx]

ax.hist(edu_boot, bins=40, density=True, alpha=0.7, color='steelblue', edgecolor='white')
ax.axvline(x=boot_results['params']['education'], color='red', linewidth=2, 
           label=f"Point Estimate: {boot_results['params']['education']:.4f}")
ax.axvline(x=boot_results['ci_lower']['education'], color='black', linestyle='--', linewidth=1.5,
           label=f"95% CI Lower: {boot_results['ci_lower']['education']:.4f}")
ax.axvline(x=boot_results['ci_upper']['education'], color='black', linestyle='--', linewidth=1.5,
           label=f"95% CI Upper: {boot_results['ci_upper']['education']:.4f}")

ax.set_xlabel('Coefficient: Education')
ax.set_ylabel('Density')
ax.set_title('Bootstrap Distribution of Returns to Education')
ax.legend()
plt.show()

In [ ]:
# Comparison: Analytical SE vs Bootstrap SE vs Robust SE
formula = 'log_wage ~ education + experience + female'
data_subset = nlsy.sample(500, random_state=42)

ols_classical = smf.ols(formula, data=data_subset).fit()
ols_robust = smf.ols(formula, data=data_subset).fit(cov_type='HC1')
ols_cluster = smf.ols(formula, data=data_subset).fit(
    cov_type='cluster', cov_kwds={'groups': data_subset['id']}
)
boot = bootstrap_ols(data_subset, formula, n_boot=500)

# Comparison table
comp_df = pd.DataFrame({
    'Classical SE': ols_classical.bse.round(4),
    'HC1 Robust SE': ols_robust.bse.round(4),
    'Cluster SE': ols_cluster.bse.round(4),
    'Bootstrap SE': boot['boot_se'].round(4),
})
print("Standard Error Comparison:")
print(comp_df)

### Key Takeaway

- Bootstrap SEs are asymptotically equivalent to analytical SEs but make fewer distributional assumptions
- The bootstrap is particularly valuable when closed-form variance formulas are unavailable
- Clustered bootstrap (resampling clusters, not individual observations) is preferred for clustered data
- At least 500-1000 bootstrap replications are recommended for reliable inference

---

## Summary

This tutorial covered the core toolkit of applied econometrics using EconomicsLab:

| Method | Key Use Case | Key Assumption |
|---|---|---|
| OLS | Baseline estimation | Exogeneity ($E[X\epsilon] = 0$) |
| IV/2SLS | Endogenous regressors | Instrument relevance + exogeneity |
| DiD | Policy evaluation | Parallel trends |
| Panel FE | Time-invariant confounding | Strict exogeneity |
| Logit/Probit | Binary outcomes | Correct distributional specification |
| RDD | Local treatment effects | No manipulation at cutoff |
| LASSO | Variable selection | Sparsity |
| Bootstrap | Inference without asymptotics | IID sampling |

### References

1. Angrist, J. D. & Pischke, J.-S. (2009). *Mostly Harmless Econometrics*. Princeton University Press.
2. Wooldridge, J. M. (2010). *Econometric Analysis of Cross Section and Panel Data*. MIT Press.
3. Cameron, A. C. & Trivedi, P. K. (2005). *Microeconometrics: Methods and Applications*. Cambridge.
4. Cunningham, S. (2021). *Causal Inference: The Mixtape*. Yale University Press.

---

**Thank you for completing the EconomicsLab tutorial!** Contributions, issues, and feedback are welcome at [github.com/econlab/econlab](https://github.com/econlab/econlab).